# Livrable 3 - Analyse massive & performance

Sur ce notebook nous allons reprendre la base du livrable 2 et l'étendre sur la France entière et sur 4 années (2021 - 2024) (Impossible de télécharger le fichier 2020, le fichier est vide 4Ko)

Nous produirons les indicateurs 1 à 4 du périmètre fonctionnel.
Une jointure avec le référentiel des communes sera ajouté.
Nous ferons également un benchmark documenté sur : 
  * sur prix moyen/m2 par département sur 1 an
  * le temps en mémoire des requêtes (afin de déterminer quand Spark est plus intéressant que Pandas)

Les résultats seront exporter dans un format en .parquet et en .csv

## Lecture des fichiers dvf France de 2021 à 2024 avec Spark

In [15]:
import time
from pyspark.sql import SparkSession

# Configuration de Spark
spark = (
    SparkSession.builder
                .appName("DateImmo_Livrable3")
                .getOrCreate()
)

print("Démarrage du chargement de Spark...")
start_time = time.time()

# Lecture des fichier dvf_France 2021 - 2024
df_dvf = spark.read.csv(
    "../data/raw/dvf_full_202*.csv.gz",
    header=True
)

# Lecture du référentiel des communes
df_communes = spark.read.csv(
    "../data/raw/communes_france.csv",
    header=True
)

total_lignes = df_dvf.count()
temps_ecoule = time.time() - start_time

print(f"Nombre total d'enregistrements fichier dvf (Spark) : {total_lignes}")
print(f"Temps de chargement de Spark : {temps_ecoule:.2f} secondes")

Démarrage du chargement de Spark...


26/09/08 21:54:46 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ../data/raw/dvf_full_202*.csv.gz.
java.io.FileNotFoundException: File ../data/raw/dvf_full_202*.csv.gz does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210)
	at org.apache.spark.sql

Nombre total d'enregistrements fichier dvf (Spark) : 16668086
Temps de chargement de Spark : 2.33 secondes


## Lecture des fichiers dvf France de 2021 à 2024 avec Pandas

In [16]:
import pandas as pd
import time

print(f"Démarrage du chargement avec Pandas")
start_time = time.time()

fichiers_dvf = [
    "../data/raw/dvf_full_2021.csv.gz",
    "../data/raw/dvf_full_2022.csv.gz",
    "../data/raw/dvf_full_2023.csv.gz",
    "../data/raw/dvf_full_2024.csv.gz"
]

# Lecture des fichier dvf, et stockage dans une liste
liste_dataframes = []
for fichier in fichiers_dvf:
    print(f"Lecture de {fichier}")
    df_temp = pd.read_csv(fichier, low_memory=False)
    liste_dataframes.append(df_temp)

df_pandas = pd.concat(liste_dataframes, ignore_index=True)

temps_ecoule_pd = time.time() - start_time

print(f"Nombre total d'enregistrement (Pandas) : {len(df_pandas)}")
print(f"Temps de chargement Pandas : {temps_ecoule_pd:.2f} secondes")

Démarrage du chargement avec Pandas
Lecture de ../data/raw/dvf_full_2021.csv.gz
Lecture de ../data/raw/dvf_full_2022.csv.gz
Lecture de ../data/raw/dvf_full_2023.csv.gz
Lecture de ../data/raw/dvf_full_2024.csv.gz
Nombre total d'enregistrement (Pandas) : 16668086
Temps de chargement Pandas : 51.18 secondes


## Nettoyage et calcul des indicateurs

In [17]:
from pyspark.sql.functions import col, round, avg, year, month, count

# Nettoyage des données

df_clean = df_dvf.dropna(subset=["valeur_fonciere", "surface_reelle_bati"])
df_clean = df_clean.withColumn(
    "prix_m2", 
    col("valeur_fonciere").cast("double") / col("surface_reelle_bati").cast("double")
)
df_clean = df_clean.filter((col("prix_m2") >= 100) & (col("prix_m2") <= 20000))

# Indicateur 1 : Prix au m2 par département
ind_dept = (
    df_clean.groupBy("code_departement")
            .agg(round(avg("prix_m2"), 2).alias("prix_moyen_m2"))
            .orderBy("code_departement")
)

# Indicateur 2 : Evolution annuelle du prix miyen
ind_annee = (
    df_clean.withColumn("annee", year("date_mutation"))
            .groupBy("annee")
            .agg(round(avg("prix_m2"), 2).alias("prix_moyen_m2"))
            .orderBy("annee")
)

# Indicateur 3 : Top communes 
ind_top_communes = (
    df_clean.groupBy("nom_commune")
            .agg(count("*").alias("volume_de_ventes"))
            .orderBy(col("volume_de_ventes").desc())
)


#Indicateur 4 : Volumes de ventes mensuels
ind_mois = (
    df_clean.withColumn("mois", month("date_mutation"))
            .groupBy("mois")
            .agg(count("*").alias("volume_de_ventes"))
            .orderBy("mois")
)

print("Les indicateurs sont préparés dans le plans d'exécution de Spark")

Les indicateurs sont préparés dans le plans d'exécution de Spark


## Déclenchement des calculs

In [18]:
print(" * Prix / m2 par departement : ")
ind_dept.show(5)

print(" * Evolution annuelle : ")
ind_annee.show()

print(" * Top 10 des communes : ")
ind_top_communes.show(10)

print(" * Volumes mensuels : ")
ind_mois.show(12)

 * Prix / m2 par departement : 


+----------------+-------------+
|code_departement|prix_moyen_m2|
+----------------+-------------+
|              01|      3232.36|
|              02|      1915.23|
|              03|       1944.3|
|              04|      2900.49|
|              05|       3292.7|
+----------------+-------------+
only showing top 5 rows
 * Evolution annuelle : 


+-----+-------------+
|annee|prix_moyen_m2|
+-----+-------------+
| 2021|      3385.21|
| 2022|      3622.26|
| 2023|      3574.93|
| 2024|      3452.32|
+-----+-------------+

 * Top 10 des communes : 


+-------------+----------------+
|  nom_commune|volume_de_ventes|
+-------------+----------------+
|     Toulouse|           40246|
|         Nice|           37972|
|       Nantes|           25410|
|  Montpellier|           24444|
|     Bordeaux|           23703|
|        Lille|           21009|
|Saint-Étienne|           20429|
|        Dijon|           17674|
|       Toulon|           16528|
|       Rennes|           16253|
+-------------+----------------+
only showing top 10 rows
 * Volumes mensuels : 


+----+----------------+
|mois|volume_de_ventes|
+----+----------------+
|   1|          368796|
|   2|          363313|
|   3|          428571|
|   4|          397549|
|   5|          411587|
|   6|          504029|
|   7|          551356|
|   8|          369279|
|   9|          480086|
|  10|          433152|
|  11|          366648|
|  12|          497454|
+----+----------------+



## Jointure des DVF avec le fichier communes

In [24]:
# Préparation du csv communes
ref_communes = df_communes.select(
    col("nom_commune").alias("nom_commune_ref"), 
    col("nom_region"),
    col("latitude"),
    col("longitude")
)

# Jointure DVF et communes

df_enrichi = df_clean.join(
    ref_communes,
    df_clean["nom_commune"] == ref_communes["nom_commune_ref"],
    how="left"
)

df_enrichi.select("date_mutation", "nom_commune", "prix_m2", "nom_region").show(10)

+-------------+--------------------+------------------+--------------------+
|date_mutation|         nom_commune|           prix_m2|          nom_region|
+-------------+--------------------+------------------+--------------------+
|   2022-01-03|     Bourg-en-Bresse|2291.6666666666665|Auvergne-Rhône-Alpes|
|   2022-01-03|     Bourg-en-Bresse|2291.6666666666665|Auvergne-Rhône-Alpes|
|   2022-01-03|           Savigneux|1021.4285714285714|Auvergne-Rhône-Alpes|
|   2022-01-03|           Savigneux|1021.4285714285714|Auvergne-Rhône-Alpes|
|   2022-01-06|    Mantenay-Montlin|2361.1111111111113|Auvergne-Rhône-Alpes|
|   2022-01-03|Saint-André-de-Corcy| 4166.666666666667|Auvergne-Rhône-Alpes|
|   2022-01-03|Saint-André-de-Corcy|1238.2075471698113|Auvergne-Rhône-Alpes|
|   2022-01-05|     Bourg-en-Bresse|  547.008547008547|Auvergne-Rhône-Alpes|
|   2022-01-05|     Bourg-en-Bresse|  547.008547008547|Auvergne-Rhône-Alpes|
|   2022-01-03|     Bourg-en-Bresse|            1400.0|Auvergne-Rhône-Alpes|

## Benchmark Pandas VS Spark sur 1 année (2023)

In [25]:
import time
import pandas as pd
from pyspark.sql.functions import col, avg, round

fichier_2023 = "../data/raw/dvf_full_2023.csv.gz"

print("Début du BENCHMARK : Prix moyen/m2 par département sur l'année 2023")

# --------------------------------------------------------------------------------------------------------------------------------------------
# Test avec Pandas
start_pd = time.time()

# Lecture
df_pd = pd.read_csv(fichier_2023, low_memory=False)

# Nettoyage 
df_pd = df_pd.dropna(subset=["valeur_fonciere", "surface_reelle_bati"])
df_pd["valeur_fonciere"] = pd.to_numeric(df_pd["valeur_fonciere"], errors='coerce')
df_pd["surface_reelle_bati"] = pd.to_numeric(df_pd["surface_reelle_bati"], errors='coerce')
df_pd["prix_m2"] = df_pd["valeur_fonciere"] / df_pd["surface_reelle_bati"]
df_pd = df_pd[(df_pd["prix_m2"] >= 100) & (df_pd["prix_m2"] <= 20000)]

result_pd = df_pd.groupby("code_departement")["prix_m2"].mean().reset_index()
# Calcul de temps
temps_pd = time.time() - start_pd
print(f"Temps total Pandas : {temps_pd:.2f} secondes")

# --------------------------------------------------------------------------------------------------------------------------------------------
# Test avec Spark
start_sp = time.time()

# Lecture
df_sp = spark.read.csv(fichier_2023, header=True)

# Nettoyage 
df_sp = df_sp.dropna(subset=["valeur_fonciere", "surface_reelle_bati"])
df_sp = df_sp.withColumn("prix_m2", col("valeur_fonciere").cast("double") / col("surface_reelle_bati").cast("double"))
df_sp = df_sp.filter((col("prix_m2") >= 100) & (col("prix_m2") <= 20000))

# Calcul 
result_sp = df_sp.groupBy("code_departement").agg(round(avg("prix_m2"), 2).alias("prix_moyen_m2"))
result_sp.collect() 
temps_sp = time.time() - start_sp
print(f"Temps total Spark : {temps_sp:.2f} secondes")

Début du BENCHMARK : Prix moyen/m2 par département sur l'année 2023
Temps total Pandas : 11.64 secondes


Temps total Spark : 4.71 secondes


## Benchmark Pandas VS Spark sur 4 années (2021 à 2024)

In [26]:
import time
import pandas as pd
from pyspark.sql.functions import col, avg, round

fichiers_multiples = [f"../data/raw/dvf_full_{annee}.csv.gz" for annee in range(2021, 2025)]

print("Début de BENCHMARK : Prix moyen/m2 par département (2021 à 2024)")


# --------------------------------------------------------------------------------------------------------------------------------------------
# Test avec Pandas
start_pd = time.time()

# Lecture en boucle et assemblage 
liste_dataframes = []
for fichier in fichiers_multiples:
    df_temp = pd.read_csv(fichier, low_memory=False)
    liste_dataframes.append(df_temp)
df_pd = pd.concat(liste_dataframes, ignore_index=True)

# Nettoyage 
df_pd = df_pd.dropna(subset=["valeur_fonciere", "surface_reelle_bati"])
df_pd["valeur_fonciere"] = pd.to_numeric(df_pd["valeur_fonciere"], errors='coerce')
df_pd["surface_reelle_bati"] = pd.to_numeric(df_pd["surface_reelle_bati"], errors='coerce')
df_pd["prix_m2"] = df_pd["valeur_fonciere"] / df_pd["surface_reelle_bati"]
df_pd = df_pd[(df_pd["prix_m2"] >= 100) & (df_pd["prix_m2"] <= 20000)]

# Calcul 
result_pd = df_pd.groupby("code_departement")["prix_m2"].mean().reset_index()
temps_pd = time.time() - start_pd
print(f"Temps total Pandas : {temps_pd:.2f} secondes")


# --------------------------------------------------------------------------------------------------------------------------------------------
# Test avec Spark
start_sp = time.time()

# Lecture directe de la liste 
df_sp = spark.read.csv(fichiers_multiples, header=True)

# Nettoyage 
df_sp = df_sp.dropna(subset=["valeur_fonciere", "surface_reelle_bati"])
df_sp = df_sp.withColumn("prix_m2", col("valeur_fonciere").cast("double") / col("surface_reelle_bati").cast("double"))
df_sp = df_sp.filter((col("prix_m2") >= 100) & (col("prix_m2") <= 20000))

# Calcul 
result_sp = df_sp.groupBy("code_departement").agg(round(avg("prix_m2"), 2).alias("prix_moyen_m2"))
result_sp.collect() 

temps_sp = time.time() - start_sp
print(f"Temps total Spark : {temps_sp:.2f} secondes")

Début de BENCHMARK : Prix moyen/m2 par département (2021 à 2024)
Temps total Pandas : 57.55 secondes


Temps total Spark : 5.64 secondes


## Export des résultats dans un fichier en .parquet et .csv

Premier export des résultats agrégés

In [32]:
from pyspark.sql.functions import year, col, avg, round, count

print(f"Préparation des données pour l'export")


# Création du tableau agrégé
df_export = df_clean.withColumn("annee", year("date_mutation"))

df_agg = (df_export.groupBy("annee", "code_departement")
                   .agg(
                       round(avg("prix_m2"), 2).alias("prix_moyen_m2"),
                       count("*").alias("volume_ventes")
                   ))


chemin_parquet = "../data/clean/dvf_agg_parquet"
chemin_csv = "../data/clean/dvf_agg_csv"

print(f"Export en cours (Parquet)")

df_agg.coalesce(1).write.mode("overwrite").parquet(chemin_parquet)

print(f"Export en cours (CSV)")

df_agg.coalesce(1).write.mode("overwrite").csv(chemin_csv)

print(f"Exportation terminée")

Préparation des données pour l'export
Export en cours (Parquet)


Export en cours (CSV)


Exportation terminée


Second export suite à la fusion des 2 csv

In [34]:
from pyspark.sql.functions import year

print(f"Préparation de l'import de la base enrichie")

df_enrichi_export = df_enrichi.withColumn("annee", year("date_mutation"))

chemin_enrichi_parquet = "../data/clean/dvf_enrichi_parquet"
chemin_enrichi_csv = "../data/clean/dvf_enrichi_csv"

print(f"Export de la base enrichie en cours (Parquet)")

df_agg.coalesce(1).write.mode("overwrite").parquet(chemin_enrichi_parquet)

print(f"Export de la base enrichie en cours (CSV)")

df_agg.coalesce(1).write.mode("overwrite").csv(chemin_enrichi_csv)


print(f"Exportation terminée")

Préparation de l'import de la base enrichie
Export de la base enrichie en cours (Parquet)


Export de la base enrichie en cours (CSV)


Exportation terminée
